# CoRD-Net — `e2_fgbf_pim_v2` Experiment on Kaggle (T4 GPU)

**Goal**: Validate the FGBF PIM-Lite *active fusion* fix (`fgbf_fuse_main=True`)  
that routes the 256-d boundary feature into the main 5-way KL classifier,  
addressing the verified KL1 misclassification failure mode.

| Setting | Value |
|---------|-------|
| Branch | `dev_fgbf` |
| Experiment | `e2_fgbf_pim_v2` |
| GPU | NVIDIA T4 |
| Loss | `weighted_ce` (class-balanced) |
| Sampler | `weighted` |
| Monitor | `0.5*macro_f1 + 0.5*kl1_f1` (composite) |
| Early stop guard | `min_epochs_before_early_stop=15` |

**Cells in order**:
1. Environment setup & checkout
2. Smoke-test: config, model construction, forward pass, loss, metrics
3. Dataset path resolution (Kaggle auto-detect)
4. Pre-training config audit
5. Train `e2_fgbf_pim_v2`
6. Evaluate on test split
7. Inspect & display results

---
## Cell 1 — Environment Setup & Codebase Checkout

In [8]:
%%bash
set -e
REPO_DIR="CoRD-Net-dev"
BRANCH="dev_fgbf"
REPO_URL="https://github.com/sanjayrk2007/CoRD-Net-dev"

if [ ! -d "$REPO_DIR/.git" ]; then
    echo "==> Cloning $REPO_URL (branch: $BRANCH) ..."
    git clone -b "$BRANCH" "$REPO_URL" "$REPO_DIR"
else
    echo "==> Repo exists. Pulling latest $BRANCH ..."
    git -C "$REPO_DIR" fetch origin
    git -C "$REPO_DIR" checkout "$BRANCH"
    git -C "$REPO_DIR" reset --hard "origin/$BRANCH"
fi

echo ""
echo "==> Commit:"
git -C "$REPO_DIR" log -1 --oneline
mkdir -p "$REPO_DIR/logs" "$REPO_DIR/checkpoints" "$REPO_DIR/results"

echo ""
echo "==> Installing dependencies ..."
pip install -q --upgrade pip
pip install -q -r "$REPO_DIR/requirements.txt"

echo ""
echo "==> PyTorch / CUDA status:"
python -c "
import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
"
nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader 2>/dev/null || true


==> Repo exists. Pulling latest dev_fgbf ...
Your branch is up to date with 'origin/dev_fgbf'.
HEAD is now at 6401a49 Rectified logical mistakes

==> Commit:
6401a49 Rectified logical mistakes

==> Installing dependencies ...

==> PyTorch / CUDA status:
PyTorch : 2.10.0+cu128
CUDA    : True
GPU     : Tesla T4
VRAM    : 15.6 GB
Tesla T4, 15360 MiB, 13453 MiB
Tesla T4, 15360 MiB, 14909 MiB


Already on 'dev_fgbf'


---
## Cell 2 — Smoke Tests (CPU, synthetic data)

Runs **before** touching any real data. Validates:
- All modules import cleanly
- `e2_fgbf_pim_v2` config flags: `fgbf_fuse_main=True`, `loss_type=weighted_ce`, `sampler=weighted`
- DRPNet forward pass shapes with `fgbf_fuse_main=True`
- FGBF gradient flows through the main classifier (not detached)
- Composite monitor score (`0.5*macro_f1 + 0.5*kl1_f1`) is computable
- `build_primary_loss('weighted_ce', ...)` returns weighted CrossEntropyLoss
- `min_epochs_before_early_stop` field exists on `TrainingConfig`
- Baseline `e2_fgbf_pim` is NOT mutated (still has `fgbf_fuse_main=False`)

All checks print PASS or FAIL.

In [9]:
import sys, os, traceback

REPO = os.path.abspath('CoRD-Net-dev')
if REPO not in sys.path:
    sys.path.insert(0, REPO)

PASS = '\033[92m  PASS\033[0m'
FAIL = '\033[91m  FAIL\033[0m'
SEP  = '=' * 68
results = []

def record(name, passed, detail=''):
    print(f"{PASS if passed else FAIL}  {name}")
    if not passed:
        print(f"       ↳ {detail}")
    results.append((name, passed))

def section(title):
    print(f'\n{SEP}\n  {title}\n{SEP}')

# ─── S1. Imports ─────────────────────────────────────────────────
section('S1. IMPORTS')

try:
    from config import get_config, ModelConfig, TrainingConfig, EXPERIMENT_NAMES, FGBF_FLAGS
    record('config.py imports cleanly', True)
except Exception as e:
    record('config.py imports cleanly', False, str(e))
    raise SystemExit('Aborting: config.py failed')

try:
    from models.drpnet import DRPNet
    record('models.drpnet imports cleanly', True)
except Exception as e:
    record('models.drpnet imports cleanly', False, str(e))
    raise SystemExit('Aborting: DRPNet failed')

try:
    from models.fgbf import FineGrainedBoundaryFeatureModule
    record('models.fgbf imports cleanly', True)
except Exception as e:
    record('models.fgbf imports cleanly', False, str(e))

try:
    from losses import build_primary_loss, MultiTaskLoss
    record('losses.py imports cleanly', True)
except Exception as e:
    record('losses.py imports cleanly', False, str(e))

try:
    from metrics import compute_all_metrics, evaluate
    record('metrics.py imports cleanly', True)
except Exception as e:
    record('metrics.py imports cleanly', False, str(e))

try:
    from trainer import Trainer
    record('trainer.py imports cleanly', True)
except Exception as e:
    record('trainer.py imports cleanly', False, str(e))

# ─── S2. Config audit ────────────────────────────────────────────
section('S2. CONFIG AUDIT — e2_fgbf_pim_v2')

cfg = get_config('e2_fgbf_pim_v2')
mc, tc = cfg.model, cfg.training

record("'e2_fgbf_pim_v2' in EXPERIMENT_NAMES",
       'e2_fgbf_pim_v2' in EXPERIMENT_NAMES)
record('model.use_stn = True', mc.use_stn == True, f'got {mc.use_stn}')
record('model.use_fgbf = True', mc.use_fgbf == True, f'got {mc.use_fgbf}')
record("model.fgbf_block = 'pim'", mc.fgbf_block == 'pim', f'got {mc.fgbf_block}')
record('model.fgbf_fuse_main = True  ← THE CRITICAL FIX',
       mc.fgbf_fuse_main == True,
       f'got {mc.fgbf_fuse_main} — fusion path is INACTIVE')
record('model.use_dual_intensity = False',
       mc.use_dual_intensity == False, f'got {mc.use_dual_intensity}')
record("training.loss_type = 'weighted_ce'",
       tc.loss_type == 'weighted_ce', f'got {tc.loss_type}')
record("training.sampler = 'weighted'",
       tc.sampler == 'weighted', f'got {tc.sampler}')
record('training.min_epochs_before_early_stop >= 15',
       hasattr(tc, 'min_epochs_before_early_stop') and tc.min_epochs_before_early_stop >= 15,
       f'got {getattr(tc, "min_epochs_before_early_stop", "MISSING")}')
record('FGBF_FLAGS: fgbf_fuse_main=True and fgbf_block=pim',
       FGBF_FLAGS.get('fgbf_fuse_main') == True and FGBF_FLAGS.get('fgbf_block') == 'pim',
       f'FGBF_FLAGS = {FGBF_FLAGS}')

# Baseline must NOT be mutated
cfg_base = get_config('e2_fgbf_pim')
record('e2_fgbf_pim (baseline) fgbf_fuse_main = False (untouched)',
       cfg_base.model.fgbf_fuse_main == False,
       f'got {cfg_base.model.fgbf_fuse_main} — baseline has been mutated!')

# ─── S3. Model construction & forward pass ───────────────────────
section('S3. MODEL CONSTRUCTION & FORWARD PASS')

import torch

try:
    model = DRPNet(get_config('e2_fgbf_pim_v2').model)
    model.eval()
    record('DRPNet constructs without error', True)
    record('model.fgbf is not None', model.fgbf is not None, 'FGBF module not instantiated')

    expected_in = cfg.model.backbone_feature_dim + cfg.model.fgbf_feature_dim
    if hasattr(model.projector, 'in_features'):
        actual_in = model.projector.in_features
        record(f'projector.in_features = {expected_in} (backbone+fgbf concatenated)',
               actual_in == expected_in,
               f'got {actual_in} — fgbf NOT fused into projector input')
    else:
        record('projector is Identity (concat_dim == fused_dim)', True)

    x = torch.randn(2, 3, 512, 512)
    with torch.no_grad():
        out = model(x)

    record("'logits' in forward output", 'logits' in out, str(list(out.keys())))
    record('logits shape = (2, 5)',
           tuple(out['logits'].shape) == (2, 5), f'got {tuple(out["logits"].shape)}')
    record("'fgbf_logits' in forward output", 'fgbf_logits' in out, str(list(out.keys())))
    record('fgbf_logits shape = (2, 3)',
           'fgbf_logits' in out and tuple(out['fgbf_logits'].shape) == (2, 3),
           f'got {tuple(out.get("fgbf_logits", torch.empty(0)).shape)}')
    record("'theta' in forward output (STN active)", 'theta' in out, str(list(out.keys())))
except Exception as e:
    record('Model construction / forward pass', False, traceback.format_exc()[-500:])

# ─── S4. Gradient flow ───────────────────────────────────────────
section('S4. GRADIENT FLOW — FGBF feature reaches main classifier')

try:
    model_g = DRPNet(get_config('e2_fgbf_pim_v2').model)
    model_g.train()
    x = torch.randn(2, 3, 512, 512)
    labels = torch.randint(0, 5, (2,))
    out = model_g(x)
    loss = torch.nn.functional.cross_entropy(out['logits'], labels)
    loss.backward()

    fgbf_grads = [p for p in model_g.fgbf.parameters()
                  if p.grad is not None and p.grad.abs().sum().item() > 0]
    record('FGBF params receive gradients from main classifier loss',
           len(fgbf_grads) > 0,
           f'0/{sum(1 for _ in model_g.fgbf.parameters())} FGBF params with grad — fusion NOT connected')

    cls_grad = model_g.classifier.weight.grad
    record('classifier.weight has non-zero gradient',
           cls_grad is not None and cls_grad.abs().sum().item() > 0,
           'classifier weight gradient is zero or None')
except Exception as e:
    record('Gradient flow test', False, traceback.format_exc()[-500:])

print(f'\n{SEP}')
passed = sum(p for _, p in results)
total  = len(results)
print(f'  SMOKE TESTS: {passed}/{total} passed')
print(SEP)



  S1. IMPORTS
  PASS  config.py imports cleanly
  PASS  models.drpnet imports cleanly
  PASS  models.fgbf imports cleanly
  PASS  losses.py imports cleanly
  PASS  metrics.py imports cleanly
  PASS  trainer.py imports cleanly

  S2. CONFIG AUDIT — e2_fgbf_pim_v2
  PASS  'e2_fgbf_pim_v2' in EXPERIMENT_NAMES
  PASS  model.use_stn = True
  PASS  model.use_fgbf = True
  PASS  model.fgbf_block = 'pim'
  PASS  model.fgbf_fuse_main = True  ← THE CRITICAL FIX
  PASS  model.use_dual_intensity = False
  PASS  training.loss_type = 'weighted_ce'
  PASS  training.sampler = 'weighted'
  PASS  training.min_epochs_before_early_stop >= 15
  PASS  FGBF_FLAGS: fgbf_fuse_main=True and fgbf_block=pim
  PASS  e2_fgbf_pim (baseline) fgbf_fuse_main = False (untouched)

  S3. MODEL CONSTRUCTION & FORWARD PASS
  PASS  DRPNet constructs without error
  PASS  model.fgbf is not None
  PASS  projector.in_features = 1024 (backbone+fgbf concatenated)
  PASS  'logits' in forward output
  PASS  logits shape = (2, 5)
 

---
## Cell 3 — Dataset Path Resolution (Kaggle Auto-Detect)

In [10]:
import os, sys
from pathlib import Path

DATA_ROOT = None

CANDIDATES = [
    "Dataset",
    "CoRD-Net-dev/Dataset",
    "/kaggle/working/CoRD-Net-dev/Dataset",
    "/kaggle/input/oai-dataset",
    "/kaggle/input/oai-knee-osteoarthritis-dataset",
    "/kaggle/input/oai-kl-grading",
    "/kaggle/input/cord-net-dataset",
    "/kaggle/input/cord-net-oai",
    "/kaggle/input/knee-osteoarthritis",
    "/kaggle/input/oai-kmri",
]

def _has_image_files(d: Path, max_check: int = 20) -> bool:
    try:
        for i, f in enumerate(d.iterdir()):
            if i >= max_check:
                break
            if f.is_file() and f.suffix.lower() in ('.png', '.jpg', '.jpeg', '.bmp'):
                return True
    except PermissionError:
        pass
    return False

def probe(root: str) -> bool:
    p = Path(root)
    if not p.exists():
        return False
    children = [c for c in p.iterdir() if c.is_dir()]
    names = {c.name for c in children}

    if "train" in names:
        grade_dirs = [c for c in (p / "train").iterdir() if c.is_dir()]
        for g in grade_dirs[:3]:
            if _has_image_files(g):
                return True

    if "0" in names:
        if _has_image_files(p / "0"):
            return True

    for child in children[:5]:
        if child.name.startswith('.'):
            continue
        sub_names = {c.name for c in child.iterdir() if c.is_dir()}
        if "train" in sub_names and _has_image_files(child / "train" / "0"):
            return True
        if "0" in sub_names and _has_image_files(child / "0"):
            return True

    return False

if DATA_ROOT is None:
    print("Scanning candidate paths for OAI dataset:")

    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        dynamic = sorted([str(c) for c in kaggle_input.iterdir() if c.is_dir()])
        CANDIDATES += [c for c in dynamic if c not in CANDIDATES]

    for cand in CANDIDATES:
        p = Path(cand)
        if not p.exists():
            print(f"  {cand}: not found")
            continue
        hit = probe(cand)
        tag = "✅ HIT" if hit else "⚠️  no images"
        try:
            tops = [c.name for c in p.iterdir() if c.is_dir()][:5]
        except Exception:
            tops = ["?"]
        print(f"  {cand}: {tag}  top-level={tops}")
        if hit and DATA_ROOT is None:
            DATA_ROOT = cand

print()
if DATA_ROOT:
    print(f"✅  DATA_ROOT = '{DATA_ROOT}'")
else:
    DATA_ROOT = "Dataset"
    print(f"Using default repo dataset path DATA_ROOT='{DATA_ROOT}'")

os.environ['DATA_ROOT'] = DATA_ROOT
Path('/tmp/cord_data_root.txt').write_text(DATA_ROOT)
print(f"   Saved to /tmp/cord_data_root.txt")


Scanning candidate paths for OAI dataset:
  Dataset: not found
  CoRD-Net-dev/Dataset: ✅ HIT  top-level=['val', 'test', 'train']
  /kaggle/working/CoRD-Net-dev/Dataset: ✅ HIT  top-level=['val', 'test', 'train']
  /kaggle/input/oai-dataset: not found
  /kaggle/input/oai-knee-osteoarthritis-dataset: not found
  /kaggle/input/oai-kl-grading: not found
  /kaggle/input/cord-net-dataset: not found
  /kaggle/input/cord-net-oai: not found
  /kaggle/input/knee-osteoarthritis: not found
  /kaggle/input/oai-kmri: not found

✅  DATA_ROOT = 'CoRD-Net-dev/Dataset'
   Saved to /tmp/cord_data_root.txt


---
## Cell 4 — Pre-Training Config Audit

Prints the full resolved config for `e2_fgbf_pim_v2` **before** training.

In [11]:
import sys, os
REPO = os.path.abspath('CoRD-Net-dev')
if REPO not in sys.path:
    sys.path.insert(0, REPO)

from config import get_config

cfg = get_config('e2_fgbf_pim_v2', pretrained=True, device='cuda', batch_size=16, epochs=60)
cfg.training.patience          = 10
cfg.training.augmentation      = 'mild'
cfg.training.data_root         = os.environ.get('DATA_ROOT', 'Dataset')

SEP = '═' * 62
print(SEP)
print(f'  Experiment : {cfg.experiment.upper()}')
print(f'  Description: {cfg.description}')
print(SEP)

mc = cfg.model
print(f"  use_stn       : {mc.use_stn}")
print(f"  use_fgbf      : {mc.use_fgbf}")
print(f"  fgbf_block    : {mc.fgbf_block}")
print(f"  fgbf_fuse_main: {mc.fgbf_fuse_main} ← CRITICAL FIX FOR KL1")
print(f"  loss_type     : {cfg.training.loss_type}")
print(f"  sampler       : {cfg.training.sampler}")
print(SEP)


══════════════════════════════════════════════════════════════
  Experiment : E2_FGBF_PIM_V2
  Description: E2 + FGBF + PIM-Lite Feature Block (Fused)
══════════════════════════════════════════════════════════════
  use_stn       : True
  use_fgbf      : True
  fgbf_block    : pim
  fgbf_fuse_main: True ← CRITICAL FIX FOR KL1
  loss_type     : weighted_ce
  sampler       : weighted
══════════════════════════════════════════════════════════════


---
## Cell 5 — Train `e2_fgbf_pim_v2` on T4 GPU

**Expected runtime**: ~25–40 min on T4 (60 epochs, batch=16, AMP).  
Early stopping fires after ≥ 15 epochs if composite score (`0.5·macro_f1 + 0.5·kl1_f1`) doesn't improve for 10 epochs.

> **Key flags that distinguish this from `e2_fgbf_pim`**:
> - `fgbf_fuse_main=True` in config — the fix being validated
> - `--loss-type weighted_ce` — class-balanced cross-entropy (KL1 upweighted)
> - `--sampler weighted` — balanced mini-batch sampling

In [ ]:
%%bash
set -e
cd CoRD-Net-dev

EXP="e2_fgbf_pim_v2"
LOG="logs/${EXP}_train.log"

DATA="Dataset"
if [ -d "Dataset" ]; then
    DATA="Dataset"
elif [ -d "CoRD-Net-dev/Dataset" ]; then
    DATA="CoRD-Net-dev/Dataset"
elif [ -n "${DATA_ROOT}" ] && [ "${DATA_ROOT}" != "/kaggle/input" ]; then
    DATA="${DATA_ROOT}"
elif [ -f /tmp/cord_data_root.txt ]; then
    DATA=$(cat /tmp/cord_data_root.txt)
fi

echo "==> Training: $EXP"
echo "    Data  : $DATA"
echo "    Log   : $LOG"
echo ""

python train.py \
    --exp          "$EXP" \
    --data-root    "$DATA" \
    --pretrained \
    --device       cuda \
    --amp \
    --epochs       60 \
    --patience     10 \
    --batch-size   16 \
    --loss-type    weighted_ce \
    --sampler      weighted \
    --augmentation mild \
    --seed         42 \
    --log-dir      logs \
    --checkpoint-dir checkpoints \
    --results-dir  results \
    2>&1 | tee "$LOG"

echo ""
echo "==> Checkpoints saved:"
ls -lh checkpoints/${EXP}*.pt 2>/dev/null || echo "  (none found)"


---
## Cell 6 — Evaluate Best Checkpoint on Test Split

In [ ]:
%%bash
set -e
cd CoRD-Net-dev

EXP="e2_fgbf_pim_v2"
LOG="logs/${EXP}_eval.log"
CKPT="checkpoints/${EXP}_best.pt"

DATA="Dataset"
if [ -d "Dataset" ]; then
    DATA="Dataset"
elif [ -d "CoRD-Net-dev/Dataset" ]; then
    DATA="CoRD-Net-dev/Dataset"
elif [ -n "${DATA_ROOT}" ] && [ "${DATA_ROOT}" != "/kaggle/input" ]; then
    DATA="${DATA_ROOT}"
elif [ -f /tmp/cord_data_root.txt ]; then
    DATA=$(cat /tmp/cord_data_root.txt)
fi

if [ ! -f "$CKPT" ]; then
    echo "⚠️ Best checkpoint not found: $CKPT"
    exit 1
fi

echo "==> Evaluating: $EXP"
echo "    Checkpoint : $CKPT"
echo "    Data root  : $DATA"
echo ""

python evaluate.py \
    --checkpoint  "$CKPT" \
    --exp         "$EXP" \
    --data-root   "$DATA" \
    --split       test \
    --device      cuda \
    --results-dir results \
    2>&1 | tee "$LOG"

echo ""
echo "==> Evaluation complete."


---
## Cell 7 — Inspect Results & Display Visualizations

In [ ]:
import json, sys, os
from pathlib import Path

try:
    from IPython.display import Image, display
    HAS_DISPLAY = True
except ImportError:
    HAS_DISPLAY = False

EXP         = 'e2_fgbf_pim_v2'
RESULTS_DIR = Path('CoRD-Net-dev/results') / EXP
LOG_DIR     = Path('CoRD-Net-dev/logs')
SEP = '═' * 62

# ── 1. Training log tail ──────────────────────────────────────────
print(f'{SEP}\n  Training Log (last 30 lines)\n{SEP}')
log_file = LOG_DIR / f'{EXP}_train.log'
if log_file.exists():
    for l in log_file.read_text().splitlines()[-30:]:
        print(l)
else:
    print(f'  Log not found: {log_file}')

# ── 2. Per-class metrics ──────────────────────────────────────────
print(f'\n{SEP}\n  Per-Class Metrics\n{SEP}')
for split in ['val', 'test']:
    pcm = RESULTS_DIR / f'per_class_metrics_{split}.json'
    if pcm.exists():
        print(f'\n[{split.upper()} split]')
        with open(pcm) as f:
            data = json.load(f)
        print(f"{'Class':<8}{'Precision':>12}{'Recall':>12}{'F1':>12}{'Support':>12}")
        for cls in ['KL0', 'KL1', 'KL2', 'KL3', 'KL4']:
            m = data.get('per_class', {}).get(cls, {})
            row = (f"{cls:<8}"
                   f"{m.get('precision', float('nan')):>12.4f}"
                   f"{m.get('recall',    float('nan')):>12.4f}"
                   f"{m.get('f1',        float('nan')):>12.4f}"
                   f"{str(m.get('support', '?')):>12}")
            if cls == 'KL1':
                print(f'\033[93m{row}  ← watch this\033[0m')
            else:
                print(row)
        kl1 = data.get('per_class', {}).get('KL1', {})
        kl1_recall = kl1.get('recall', float('nan'))
        kl1_f1     = kl1.get('f1',     float('nan'))
        target_ok  = kl1_recall > 0.35
        print(f"\n  KL1 Recall = {kl1_recall:.4f}  {'✅ > 0.35' if target_ok else '❌ <= 0.35 (KL1 still under-recalled)'}")
        print(f"  KL1 F1     = {kl1_f1:.4f}")

# ── 3. Checkpoint summary ─────────────────────────────────────────
print(f'\n{SEP}\n  Checkpoint Summary\n{SEP}')
import torch
ckpt_path = Path(f'CoRD-Net-dev/checkpoints/{EXP}_best.pt')
if ckpt_path.exists():
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    print(f"  Experiment  : {ckpt.get('experiment', '?')}")
    print(f"  Best epoch  : {ckpt.get('epoch', '?')}")
    print(f"  Best score  : {ckpt.get('best_score', float('nan')):.4f}  (0.5*macro_f1 + 0.5*kl1_f1)")
    print(f"  Best QWK    : {ckpt.get('best_qwk', float('nan')):.4f}")
else:
    print(f'  Checkpoint not found: {ckpt_path}')

print(f'\n{SEP}')
print('  Done. Check KL1 recall/F1 above to verify the fusion fix worked.')
print(SEP)
